## 🧪 Interactive Sandbox
Run the setup cell, then try any query from this chapter with `run("...")`.

In [ ]:
# ── SQL Sandbox — run this cell first ──────────────────────────────
# Creates an in-memory database with the sample tables so you can run the
# queries in this chapter. Use run("SELECT ...") and it returns a DataFrame.
# (SQLite supports SELECT/JOIN/GROUP BY/CTEs/window functions. A few Snowflake-
#  only bits — QUALIFY, DATE_TRUNC, SPLIT_PART — won't run here; those are noted.)
import sqlite3, pandas as pd
conn = sqlite3.connect(":memory:")
conn.executescript("""
CREATE TABLE departments (dept_id INT, dept_name TEXT, location TEXT);
CREATE TABLE employees (emp_id INT, name TEXT, dept_id INT, manager_id INT,
                        salary INT, hire_date TEXT, email TEXT);
CREATE TABLE customers (customer_id INT, name TEXT, city TEXT, signup_date TEXT);
CREATE TABLE products (product_id INT, product_name TEXT, category TEXT, price REAL);
CREATE TABLE orders (order_id INT, customer_id INT, product_id INT, quantity INT,
                     order_date TEXT, amount REAL);
INSERT INTO departments VALUES
 (1,'Engineering','Bangalore'),(2,'Sales','Mumbai'),(3,'Marketing','Delhi'),(4,'HR','Bangalore');
INSERT INTO employees VALUES
 (1,'Asha',1,NULL,180000,'2019-03-01','asha@pjs.com'),
 (2,'Ravi',1,1,120000,'2020-06-15','ravi@pjs.com'),
 (3,'Meera',2,1,95000,'2021-01-20','meera@pjs.com'),
 (4,'Kiran',2,3,70000,'2022-07-11','kiran@pjs.com'),
 (5,'Sana',3,1,88000,'2021-09-05','sana@pjs.com'),
 (6,'Vijay',1,2,110000,'2023-02-28','vijay@pjs.com'),
 (7,'Nina',4,1,60000,'2020-11-30',NULL);
INSERT INTO customers VALUES
 (101,'Rahul','Mumbai','2023-01-10'),(102,'Priya','Delhi','2023-02-14'),
 (103,'Arjun','Bangalore','2023-03-22'),(104,'Divya','Mumbai','2023-05-01');
INSERT INTO products VALUES
 (201,'Laptop','Electronics',55000),(202,'Mouse','Electronics',700),
 (203,'Desk','Furniture',8000),(204,'Chair','Furniture',4500),(205,'Monitor','Electronics',15000);
INSERT INTO orders VALUES
 (1,101,201,1,'2023-06-01',55000),(2,101,202,2,'2023-06-01',1400),
 (3,102,205,2,'2023-06-05',30000),(4,103,203,1,'2023-06-10',8000),
 (5,103,204,4,'2023-06-10',18000),(6,104,201,1,'2023-07-02',55000),
 (7,102,202,1,'2023-07-15',700),(8,101,205,1,'2023-08-01',15000);
""")
def run(sql):
    "Run a SQL query against the sample database and return a DataFrame."
    return pd.read_sql_query(sql, conn)

run("SELECT * FROM employees")  # try it — edit and re-run!

# Module 04 — JOINs: Combining Tables

> The single most important SQL skill. Real data is split across tables; JOINs put it back together. Interviews live here.

---

## 4.1 Why Tables Are Split (Normalization)

We don't store the department name in every employee row — we'd repeat it thousands of times. Instead, `employees` stores a `dept_id` that **points to** the `departments` table. JOINs follow those pointers.

```
employees                departments
emp_id dept_id           dept_id  dept_name
  1      1     ────────►    1      Engineering
  2      1                  2      Sales
  3      2     ────────►
```

## 4.2 INNER JOIN — Matching Rows from Both Tables

Returns rows where the join condition matches in **both** tables:

```sql
SELECT e.name, d.dept_name
FROM employees e
JOIN departments d ON e.dept_id = d.dept_id;
```

- `e` and `d` are **aliases** (shorthand for the tables).
- `ON e.dept_id = d.dept_id` is the **join condition** (which rows pair up).
- Plain `JOIN` = `INNER JOIN`.

> An employee with a `dept_id` that has no match in `departments` is **dropped** by an inner join.

## 4.3 LEFT JOIN — Keep All Left Rows

Keeps **every** row from the left table; unmatched right-side columns become `NULL`:

```sql
-- All departments, even those with no employees
SELECT d.dept_name, e.name
FROM departments d
LEFT JOIN employees e ON d.dept_id = e.dept_id;
```

Use `LEFT JOIN` when you must not lose rows from the main table.

```mermaid
flowchart LR
    L[LEFT: all rows kept] -->|match| M[matched pairs]
    L -->|no match| N[right cols = NULL]
```

## 4.4 The Join Types

| Join | Keeps |
|------|-------|
| `INNER JOIN` | only matching rows (both sides) |
| `LEFT JOIN` | all left rows + matches |
| `RIGHT JOIN` | all right rows + matches |
| `FULL OUTER JOIN` | all rows from both sides |
| `CROSS JOIN` | every combination (Cartesian product) |

`RIGHT JOIN` is just a `LEFT JOIN` written backwards — most people always use LEFT.

## 4.5 The Anti-Join — "Rows With No Match"

A super-common interview task: find rows in A with **no** match in B. Do a `LEFT JOIN` and keep the NULLs:

```sql
-- Customers who never placed an order
SELECT c.name
FROM customers c
LEFT JOIN orders o ON c.customer_id = o.customer_id
WHERE o.order_id IS NULL;      -- no matching order → NULL
```

## 4.6 Joining Three or More Tables

Chain joins — each `JOIN … ON` links one more table:

```sql
SELECT o.order_id, c.name AS customer, p.product_name, o.amount
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
JOIN products  p ON o.product_id  = p.product_id;
```

## 4.7 Self Join — a Table Joined to Itself

When rows relate to other rows in the *same* table (e.g., employee → manager):

```sql
SELECT e.name AS employee, m.name AS manager
FROM employees e
JOIN employees m ON e.manager_id = m.emp_id;   -- m = the manager row
```

Treat the two aliases (`e`, `m`) as if they were two separate tables.

## 4.8 JOIN + GROUP BY — the Everyday Combo

Most analytics = join tables, then aggregate:

```sql
-- Revenue per product category
SELECT p.category, SUM(o.amount) AS revenue
FROM orders o
JOIN products p ON o.product_id = p.product_id
GROUP BY p.category
ORDER BY revenue DESC;
```

## 4.9 Common JOIN Mistakes

- **Forgetting the ON clause** → accidental CROSS JOIN (every combination — huge, wrong results).
- **Wrong join type** → an INNER JOIN silently drops rows you needed (use LEFT).
- **Counting after a join** → `COUNT(*)` may double-count if one row matches many; use `COUNT(DISTINCT …)`.
- **Ambiguous columns** → if both tables have `dept_id`, qualify it: `e.dept_id`.

## 4.10 A Mental Model

Think of a JOIN as: "for each row on the left, find the matching row(s) on the right and stick their columns on." INNER drops non-matches; LEFT keeps them with NULLs.

---

## ✅ Key Takeaways
1. Tables are split to avoid repetition; **JOINs recombine them** via key columns.
2. `INNER JOIN` keeps only matches; `LEFT JOIN` keeps all left rows (+NULLs for non-matches).
3. **Anti-join** = `LEFT JOIN … WHERE right.key IS NULL` → "rows with no match".
4. Chain joins for 3+ tables; use a **self join** for within-table relationships (employee↔manager).
5. **JOIN + GROUP BY** powers most analytics.
6. Qualify columns (`e.dept_id`), pick the right join type, and beware double-counting.

## 🏋️ Exercises
1. List each employee with their department name (INNER JOIN).
2. List all departments and their employees, including empty departments (LEFT JOIN).
3. Find customers who never placed an order (anti-join).
4. Show each order with the customer name and product name (3-table join).
5. Show each employee alongside their manager's name (self join).
6. Compute total revenue per product category (join + group by).

**Next:** [Module 05 — Subqueries & CTEs →](module-05-subqueries-ctes.md)

---

*🗄️ SQL Mastery — [PJ's Academy](https://pjsacademy.com)*